# JsonOutputParser → `with_structured_output()` (+ 여전히 유효한 사용처)

`JsonOutputParser`는 `langchain_core`에 그대로 있으며 폐기되지 않았습니다. 하지만 역할이 달라졌습니다.

| 상황 | 현재 권장 |
|---|---|
| 스키마가 정해져 있고, 모델이 구조화 출력을 지원 | `llm.with_structured_output(Schema)` |
| 스키마 없이 "아무 JSON이나" 받고 싶음 | `llm.with_structured_output(method="json_mode")` |
| 네이티브 구조화 출력을 지원하지 않는 모델 / 일반 텍스트 응답 속 JSON 추출 | `JsonOutputParser` |

`JSON` 자체의 기본 구조(객체 `{}` / 배열 `[]`, 키-값 쌍)에 대한 설명은 책의 내용과 동일합니다.

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

## 1. 스키마 정의

책에서는 `hashtags: str`로 정의했지만, 여러 개의 키워드는 `list[str]`로 정의하는 편이 후처리에 훨씬 유리합니다.

In [ ]:
from pydantic import BaseModel, Field


class Topic(BaseModel):
    """주제에 대한 설명과 키워드"""

    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: list[str] = Field(description="해시태그 형식의 키워드(2개 이상), 예: #기후변화")

## 2. 현재 권장 방식: `with_structured_output()`

- Pydantic 클래스를 넘기면 Pydantic 객체가,
- JSON이 담긴 `dict`가 필요하면 `.model_dump()`로 변환하거나 TypedDict/JSON Schema를 스키마로 넘기면 됩니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

question = "지구 온난화의 심각성 대해 알려주세요."

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("human", "#Question: {question}"),
    ]
)

chain = prompt | llm.with_structured_output(Topic)

answer = chain.invoke({"question": question})
print(type(answer))
answer.model_dump()

JSON Schema(dict)를 직접 넘기면 결과도 `dict`로 받습니다. JSON Schema에는 최상위 `title`, `description`이 있어야 합니다.
Pydantic 모델에서 JSON Schema를 뽑아 쓸 수도 있습니다: `Topic.model_json_schema()`.

In [ ]:
topic_schema = {
    "title": "Topic",
    "description": "주제에 대한 설명과 키워드",
    "type": "object",
    "properties": {
        "description": {"type": "string", "description": "주제에 대한 간결한 설명"},
        "hashtags": {
            "type": "array",
            "items": {"type": "string"},
            "description": "해시태그 형식의 키워드(2개 이상)",
        },
    },
    "required": ["description", "hashtags"],
}

dict_chain = prompt | llm.with_structured_output(topic_schema)
response = dict_chain.invoke({"question": question})
print(type(response))
response

## 3. 스키마 없이 JSON 받기: `method="json_mode"`

책의 "Pydantic 없이 `JsonOutputParser` 사용" 예제에 해당합니다. JSON 모드는 **유효한 JSON**만 보장하고 키 구조는 보장하지 않으므로, 원하는 키를 프롬프트에 명시해야 합니다. (OpenAI JSON 모드는 프롬프트에 "JSON"이라는 단어가 포함되어야 합니다.)

In [ ]:
json_llm = llm.with_structured_output(method="json_mode")

question2 = (
    "지구 온난화에 대해 알려주세요. JSON 으로 답하세요. "
    "온난화에 대한 설명은 `description`에, 관련 키워드는 `hashtags`(배열)에 담아주세요."
)

response = (prompt | json_llm).invoke({"question": question2})
print(response)

## 4. `JsonOutputParser`가 여전히 유용한 경우

네이티브 구조화 출력을 지원하지 않는 모델을 쓰거나, 일반 텍스트 응답에서 JSON을 추출해야 할 때입니다.
`JsonOutputParser`는 **불완전한 JSON도 부분 파싱**하므로 스트리밍 시 점점 채워지는 dict를 받을 수 있습니다.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(pydantic_object=Topic)

parser_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("human", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

parser_chain = parser_prompt | llm | parser

for partial in parser_chain.stream({"question": question}):
    print(partial)